# Lesson 6.1: The Logic of Hypothesis Testing

Find out how surprising 60 heads in 100 flips would be if the coin were fair, first by simulation and then with the exact binomial test.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A friend hands you a coin, you flip it 100 times and get 60 heads. You say it is rigged, your friend says it is fair and 60 is just luck. We have one data set: 60 heads in 100 flips. Does that result give us good evidence the coin is not fair, or is it the kind of thing a fair coin does fairly often?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Set up the coin result

Our data: 60 heads in 100 flips. The null hypothesis H0 says the coin is fair (chance of heads = 0.5). The alternative H1 says it is not fair (chance of heads is not 0.5, too many or too few). NumPy makes pretend coins and SciPy gives the exact test.

In [1]:
import numpy as np  # Load NumPy, the library for fast number work, under the short name np so we can write np. instead of numpy.
from scipy import stats  # Load SciPy's statistics tools; the exact binomial test we need lives in stats.

flips = 100  # Store the number of flips under a name, so we reuse it instead of typing 100 in many places.
heads = 60  # Store what we saw: 60 heads. This is the one data point the whole lab is about.
rng = np.random.default_rng(42)   # seeded, so the result repeats  # Create a random number generator. The seed 42 makes the 'random' coins repeat, so everyone gets the same numbers.

print('observed:', heads, 'heads in', flips, 'flips')  # Print the data in words as a check that we typed the right values before analysing anything.

observed: 60 heads in 100 flips


**What this means.** This step only sets the scene: 60 heads in 100 flips, which is 10 more than the 50 a fair coin gives on average. Nothing is tested yet. The real question is whether a gap of 10 is big or small for a fair coin.

## Step 2: Simulate a fair coin

We pretend the null is true and run 100,000 experiments of 100 fair flips. The p-value is the share of experiments at least as lopsided as ours, meaning 10 or more heads away from 50 in either direction.

In [2]:
sim = rng.binomial(n=flips, p=0.5, size=100_000)   # heads in each pretend experiment  # Pretend H0 is true: run 100,000 experiments of 100 fair flips and record the heads in each one. This shows how a fair coin behaves.
gap = abs(heads - 50)                              # our result is 10 away from 50  # Our result is 10 heads away from the fair average of 50. We measure the gap so we can ask how often a fair coin does that.
extreme = np.abs(sim - 50) >= gap                  # too many OR too few heads  # Mark each experiment that is at least that far from 50, in EITHER direction, because H1 says 'not fair', not 'too many heads'.
p_sim = extreme.mean()  # The mean of True/False marks is the share of extreme experiments. That share is the simulated p-value.

print(f'average heads in the simulation: {sim.mean():.2f}')  # Print the average heads as a sanity check: it should be close to 50 if the pretend coin is fair.
print(f'simulated p-value = {p_sim:.4f}')  # Print the simulated p-value, rounded to 4 decimals so it is easy to compare with the exact one.

average heads in the simulation: 50.02
simulated p-value = 0.0571


**What this means.** In 100,000 pretend fair-coin experiments the average was 50.02 heads, so the simulation behaves like a fair coin. About 5.71% of them (p = 0.0571) were 10 or more heads away from 50, so a fair coin does something this lopsided roughly 1 time in 18. That is not rare.

## Step 3: Get the exact p-value

binomtest adds up the exact binomial chances, so no simulation is needed. The two-sided option counts both too many and too few heads. The simulated number should land close to the exact one.

In [3]:
result = stats.binomtest(heads, n=flips, p=0.5, alternative='two-sided')  # binomtest adds up the exact binomial chances. We give the successes, the flips and the fair-coin chance 0.5. 'two-sided' counts too many AND too few heads.
ci = result.proportion_ci(confidence_level=0.95)  # Ask for a 95% confidence interval for the true heads rate. It shows which rates of heads are plausible given our data.

print(f'exact p-value = {result.pvalue:.4f}')  # Print the exact p-value. Exact means no random noise, so it is the number to report.
print(f'95% CI for the heads rate: ({ci.low:.3f}, {ci.high:.3f})')  # Print the interval ends, so we can see whether 0.5 (a fair coin) is inside the plausible range.

exact p-value = 0.0569
95% CI for the heads rate: (0.497, 0.697)


**What this means.** The exact p-value is 0.0569, almost the same as the simulation (0.0571). It says: if the coin were fair, results at least this lopsided would happen about 5.7% of the time. It does not say there is a 5.7% chance the coin is fair. The 95% interval, 0.497 to 0.697, includes 0.5.

## Step 4: Make the decision

Compare p with alpha = 0.05 and word the result carefully. The normal-curve shortcut (z = 2) gives a smaller p, which would wrongly say 'significant'. This shows why the method matters when a result sits near the cut-off.

In [4]:
alpha = 0.05  # Set the surprise threshold alpha = 0.05 before deciding, so the rule is fixed and not chosen to fit the answer.
z = (heads - 50) / np.sqrt(flips * 0.5 * 0.5)  # The normal-curve shortcut: how many standard errors 60 is from 50. The standard deviation of heads in 100 fair flips is sqrt(100 x 0.5 x 0.5) = 5.
p_normal = 2 * stats.norm.sf(z)  # Turn that z into a two-sided p-value using the normal curve. sf is the area in the upper tail, and we double it for both tails.

print(f'normal shortcut: z = {z:.1f}, p = {p_normal:.4f}')  # Print the shortcut's z and p so we can compare it with the exact answer.
print('exact p <= alpha?', result.pvalue <= alpha)  # Check the decision rule on the exact p-value: is p less than or equal to alpha?
if result.pvalue <= alpha:  # Start the decision: if p is at or below alpha, we have enough evidence to reject H0.
    print('Reject H0: evidence the coin is not fair.')  # This wording is only for the 'reject' case: we say there is evidence the coin is not fair.
else:  # Otherwise (p above alpha) we do not have enough evidence against H0.
    print('Fail to reject H0: not enough evidence the coin is unfair.')  # Say 'fail to reject', never 'the coin is fair': weak evidence leaves H0 standing but does not prove it.

normal shortcut: z = 2.0, p = 0.0455
exact p <= alpha? False
Fail to reject H0: not enough evidence the coin is unfair.


**What this means.** The normal shortcut gives z = 2.0 and p = 0.0455, below 0.05, and would wrongly call the coin unfair. The exact p of 0.0569 is above 0.05, so we fail to reject H0. That is not proof the coin is fair: 100 flips are simply not quite enough. A result this close to 0.05 is a signal to collect more data.

## So, what is the answer?

With 60 heads in 100 flips the exact p-value is 0.0569, just above 0.05, so we do not have enough evidence to call the coin unfair. Caution: this does not prove the coin is fair. 100 flips can only reveal a large bias, and a result this near the cut-off is a reason to flip more, not a final verdict.

## Now you try

A seed company says 80% of its sunflower seeds germinate. You plant 50 seeds on damp paper and record 1 if a seed sprouts and 0 if it does not. Is the company's claim believable, or is the true rate different from 80%? Use the same two-sided exact test as in the coin lab.

**Data:** seeds = [1,1,0,1,1,0,1,1,1,0,1,0,1,1,1,0,0,1,1,1,0,1,1,0,1,1,0,1,0,1,1,1,0,1,1,0,1,0,1,1,1,0,1,1,0,1,1,0,1,0]

1. How many seeds germinated, and what is the sample rate?
2. Write H0 and H1, then find the exact two-sided p-value for H0: p = 0.80.
3. Check it with a simulation of 100,000 pretend experiments (np.random.default_rng(1)).
4. Decide at alpha = 0.05 and word the conclusion carefully.

*Hint:* Count the 1s with sum(). Use stats.binomtest(k, n, p=0.80). For the simulation, count how many pretend results are as far from the expected 40 as yours, in either direction.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats

seeds = [1,1,0,1,1,0,1,1,1,0,1,0,1,1,1,0,0,1,1,1,0,1,1,0,1,1,0,1,0,1,1,1,0,1,1,0,1,0,1,1,1,0,1,1,0,1,1,0,1,0]
n, k = len(seeds), sum(seeds)
res = stats.binomtest(k, n=n, p=0.80, alternative='two-sided')
print(n, 'seeds,', k, 'germinated, rate =', k / n)
print(f'exact p = {res.pvalue:.4f}')
rng = np.random.default_rng(1)
sim = rng.binomial(n, 0.80, size=100_000)
print(f'simulated p = {(np.abs(sim - 40) >= abs(k - 40)).mean():.4f}')
print('reject H0 at 0.05?', res.pvalue <= 0.05)

50 seeds, 33 germinated, rate = 0.66
exact p = 0.0201
simulated p = 0.0200
reject H0 at 0.05? True


**Answer.** 50 seeds, 33 germinated, so the sample rate is 0.66. The exact p-value is 0.0201 and the simulation gives 0.0200, both below 0.05, so we reject H0: the true germination rate is probably not 80%. The sample rate is lower than the claim, so the evidence points to fewer seeds sprouting. The test cannot say why.

## The same question in R, SPSS and Stata

**R**

```r
binom.test(60, 100, p = 0.5)  # binom.test runs the exact binomial test: 60 heads in 100 trials against a fair coin (p = 0.5). It prints the p-value and a 95% interval.

# the simulation (random numbers, so close to but not exactly the Python value)  # A comment (R ignores it). It warns that the next lines use random numbers, so their result is close to Python's but not identical.
set.seed(42)  # set.seed fixes R's random numbers so the simulation can be repeated. R's generator differs from Python's, which is why results differ slightly.
sim <- rbinom(100000, size = 100, prob = 0.5)  # rbinom makes 100,000 pretend experiments of 100 fair flips and stores the heads of each one.
mean(abs(sim - 50) >= 10)  # abs(sim - 50) >= 10 marks experiments at least 10 heads from 50 either way. mean() turns TRUE/FALSE into the simulated p-value.
```

How to read it: Look at p-value = 0.05689: if the coin were fair, results this lopsided happen about 5.7% of the time. That is above 0.05, so we fail to reject H0. The interval 0.4972 to 0.6967 contains 0.5. The last line should print about 0.057.

**SPSS** (Analyze > Nonparametric Tests > Legacy Dialogs > Binomial)

```
DATA LIST FREE / heads count.
BEGIN DATA
1 60
0 40
END DATA.
WEIGHT BY count.
NPAR TESTS
  /BINOMIAL (0.5)=heads.
```

How to read it: In the Binomial Test table, find the Sig. (2-tailed) value: about 0.057, above 0.05, so we fail to reject that the coin is fair. The observed proportions should be 0.60 and 0.40. SPSS may show an approximate p rather than the exact 0.0569.

**Stata**

```
bitesti 100 60 0.5
```

How to read it: Find the Two-sided line, Pr(k <= 40 or k >= 60) = 0.056889: the exact p-value, above 0.05, so we fail to reject H0. The one-sided Pr(k >= 60) = 0.028444 is half of it, and Stata also shows the observed proportion 0.6.